# 05 · Migración a MongoDB

**Etapa 1 del pipeline del Proyecto 2:** ETL del corpus del Proyecto 1 a MongoDB.

- Base de datos: `resennas_cine` · Colección: `resennas`
- Cada ejecución **borra la colección y la vuelve a cargar completa** desde los archivos del proyecto.



In [1]:
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto: el notebook se ejecuta desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import db_manager as dbm

print("MongoDB:", dbm.MONGO_URI, "·", dbm.NOMBRE_BD, "/", dbm.NOMBRE_COLECCION)

MongoDB: mongodb://localhost:27017 · resennas_cine / resennas


## 1. Esquema del documento

Se respeta el esquema de la sección 3.1 del enunciado. Además se guarda `id_resena`, el `id` del Proyecto 1, para unir cada documento con `pos_analysis.csv`, las métricas y los embeddings.

Decisiones para las reseñas del Proyecto 1:

| Campo | Valor |
|---|---|
| `origen_texto` | `"traducido"` |
| `tipo_obra` | `"pelicula"` (el dataset de IMDb solo tiene películas) |
| `genero_principal` | género asignado en el Proyecto 1, traducido al español (Action → Acción, Horror → Terror, …) |
| `generos`, `titulo_obra`, `fecha_recopilacion` | `null`: no están en `resenas_limpias.csv` |
| `polaridad` | columna `valoracion` (1-4 negativa, 5-6 intermedia, 7-10 positiva) |
| `periodo` | 1998-2009 → `antes_2010`, 2010-2018 → `2010_en_adelante` |
| `fuente`, `url_fuente` | `"imdb_kaggle"` y la página de la película en IMDb |
| `metricas.num_palabras` | palabras del texto en español (`n_palabras_es`) |
| `metricas.densidad_lexica` | en %, igual que en el Proyecto 1 |

## 2. Migración

In [ ]:
%pip install --upgrade pymongo

In [2]:
resumen = dbm.migrar()
pd.Series(resumen)

reseñas cargadas           16991
con pos_tags               16991
con métricas               16991
con embeddings Word2Vec        0
con embeddings BETO            0
nativas (scraping)             0
dtype: int64

## 3. Verificación

Un documento de ejemplo (sin los tokens ni los embeddings, que son largos):

In [3]:
coleccion = dbm.obtener_coleccion()

ejemplo = coleccion.find_one({}, {"pos_tags": 0, "embeddings": 0})
ejemplo

{'_id': ObjectId('6aef54d0975d94f50abf389e'),
 'id_resena': 1,
 'texto_es': 'Lo bueno: La comedia, Drama, Psicología, Filosofía, Actuar, dirección estelar, grandes personajes, violencia arenosa Lo malo: Gráfico... espera, ¿es eso algo malo?? Conclusión: A los hombres les encantará esta película. Míralo. Si no eres demasiado sensible o femenina. Esta película es una que dejará una impresión permanente. Su parte comedia anarquista, pensamiento intelectual, película de boxeo Gritty, estudio de carácter psicológico, comentario social y romance. Este primer dibujo de películas es lo bien dirigido que es. Las tomas son simplemente hermosas, mostrando al mundo en todos sus detalles arenosos y sin tener reparos en romper la cuarta pared. No voy a hablar de la trama en detalle, principalmente porque contiene tanto detalle y profundidad que simplemente estaría haciendo un flaco servicio. Baste decir que este es de lejos el concepto más creativo que he visto desde A Clockwork Orange. Realmente no